# combine

> Combine calibrated dithers into a single mosaic using swarp.

In [ ]:
# | default_exp euclid.combine

In [ ]:
# | exporti

import tempfile
from pathlib import Path
from abc import ABC, abstractmethod
import subprocess

import numpy as np
from astropy.io import fits
from astropy.stats import SigmaClip
from astropy import units as u
from astropy.coordinates import SkyCoord
from photutils.background import Background2D

from nicl.euclid.data_access import DataAccess
from nicl.mask import fast_mask
from nicl.euclid.constants import VIS, NISP, SWARP_CONFIG_NISP, SWARP_CONFIG_VIS
from nicl.euclid.utilities import default_data_path, round_up_box_size

In [ ]:
# | hide
# additional imports for examples

from nicl.utilities import physical_to_angular

In [ ]:
# | export
# base class for combining images


class Combiner(ABC):
    """Combines Euclid images using SWarp"""

    def __init__(
        self,
        in_dir=None,  # directory at which to find the input images
        out_dir=None,  # directory at which to place output images
        filters=None,  # list of filters to combine
        obs_ids=None,  # list of obs_ids to combine
        cutout_cen=None,  # central coordinates of the cutout
        cutout_size=None,  # size of the cutout in angular units
        name=None,  # suffix for the output file basename.
        bkg_sub=True,  # to subtract background or not
        bkg_mesh_size=None,  # size of the background mesh boxes in angular units
        filter_size=3,  # median filter background over `filter_size` x `filter_size` boxes
        release_name="Q1_R1",  # the data release to use to find obs_ids if required
        instrument=None,  # Euclid instrument,e.g., VIS or NISP in nicl.euclid.constants
        swarp_config=None,  # SWarp configuration file content
        overwrite=False,  # overwrite existing combined image files
        debug=False,  # retain intermediate files for checking
        **kwargs,  # command line arguments to pass to SWarp, will override the defaults
    ):
        """
        Initialize the Combine class.

        Parameters:
            in_dir (str, optional): if not specified it will use the default data path based on release_name
            out_dir (str, optional): if not specified it will use the default test data output dir based on release_name
            filters (list, optional): if not specified it will default to all filters available for a given instrument
            obs_ids (list, optional): this or cutout_cen and cutout_size must be specified. if obs_ids is not provided, they will be inferred from cutout_cen and cutout_size.
            cutout_cen (atsropy SkyCoord or str, optional): SkyCoord object or a string in the format "ra dec" (e.g. "00:00:00.0 +00:00:00.0")
            cutout_size (tuple, optional): single or a tuple of astropy Quantity object in angular units (e.g. (1*u.arcmin, 1*u.arcmin))
            name (str, optional): suffix for the output file basename. Default is a string of all obs_ids chained.
            bkg_sub (bool, optional): Whether to subtract background or not. Default is True.
            bkg_mesh_size (float, optional): single or a tuple of astropy Quantity object in angular units (e.g. (10*u.arcsec, 10*u.arcsec))
            filter_size (int, optional): Median filter background over `filter_size` x `filter_size` boxes. Default is 3.
            release_name (str, optional): The data release to use to find data if required. Default is "Q1_R1" for the current data release.
            instrument (Euclid Constant, optional): Euclid instrument, e.g., VIS or NISP in nicl.euclid.constants. Default is None.
            swarp_config (str, optional): SWarp configuration file content. Default is None.
            overwrite (bool, optional): Whether to overwrite existing combined image files. Default is False.
            debug (bool, optional): Whether to retain intermediate files for checking. Default is False.
        """
        # TODO: initialization is lengthy, should be refactored in the future
        self.instrument = instrument
        self.swarp_config = swarp_config
        self.bkg_sub = bkg_sub
        self.filter_size = filter_size
        self.release_name = release_name
        self.overwrite = overwrite
        self.debug = debug
        if in_dir is None:
            if self.release_name is not None:
                self.in_path = default_data_path(release_name)
            else:
                raise ValueError(
                    "Either in_path or release_name must be specified to find the input images."
                )
        else:
            in_dir = Path(in_dir).expanduser()
            if in_dir.is_dir() and in_dir.exists():
                self.in_path = in_dir
            else:
                raise ValueError("The input path does not exist or is not a directory.")
        if cutout_cen is not None:
            if isinstance(cutout_cen, str):
                self.cutout_cen = SkyCoord(cutout_cen, unit=(u.hourangle, u.deg))
            elif isinstance(cutout_cen, SkyCoord):
                self.cutout_cen = cutout_cen
            else:
                raise ValueError(
                    "cutout_cen must be a string in the format 'ra dec' or an astropy.coordinates.SkyCoord object."
                )
        else:
            self.cutout_cen = None
        if cutout_size is not None:
            if isinstance(cutout_size, (int, float)):
                self.cutout_size = (cutout_size * u.arcsec, cutout_size * u.arcsec)
            if isinstance(cutout_size, u.Quantity) and cutout_size.unit.is_equivalent(
                u.arcsec
            ):
                self.cutout_size = (cutout_size, cutout_size)
            elif len(cutout_size) == 2:
                if isinstance(cutout_size[0], (int, float)) and isinstance(
                    cutout_size[1], (int, float)
                ):
                    self.cutout_size = (
                        cutout_size[0] * u.arcsec,
                        cutout_size[1] * u.arcsec,
                    )
                elif (
                    isinstance(cutout_size[0], u.Quantity)
                    and cutout_size[0].unit.is_equivalent(u.arcsec)
                    and isinstance(cutout_size[1], u.Quantity)
                    and cutout_size[1].unit.is_equivalent(u.arcsec)
                ):
                    self.cutout_size = cutout_size
                else:
                    raise ValueError(
                        "cutout_size is inhomogeneous. it should be either plain numbers or astropy.units.Quantity."
                    )
            else:
                raise ValueError(
                    "cutout_size must be a single quantity or a tuple of two quantities."
                )
        else:
            self.cutout_size = None
        if (
            obs_ids is None
            and self.cutout_cen is not None
            and self.cutout_size is not None
        ):
            self.obs_ids = self._get_obs_ids()
        else:
            self.obs_ids = obs_ids
        if self.obs_ids is None or len(self.obs_ids) == 0:
            raise ValueError(
                "No observations found for the given cutout or no OBSIDs are provided."
            )
        if name is None and len(obs_ids) >= 1:
            self.name = "-".join([str(obs_id) for obs_id in obs_ids])
        else:
            self.name = name
        if out_dir is None:
            if self.release_name is not None:
                self.out_dir = default_data_path(
                    f"{self.release_name}_clusters_test", name
                )
            else:
                raise ValueError(
                    "Either out_dir or release_name must be specified to infer where to place the output images."
                )
        else:
            out_dir = Path(out_dir).expanduser()
            if out_dir.exists() and out_dir.is_file():
                raise ValueError("The output directory points to a file.")
            self.out_dir = out_dir
        if bkg_mesh_size is not None:
            if isinstance(bkg_mesh_size, (int, float)):
                self.bkg_mesh_size = (
                    bkg_mesh_size * u.arcsec,
                    bkg_mesh_size * u.arcsec,
                )
            if isinstance(
                bkg_mesh_size, u.Quantity
            ) and bkg_mesh_size.unit.is_equivalent(u.arcsec):
                self.bkg_mesh_size = (bkg_mesh_size, bkg_mesh_size)
            elif len(bkg_mesh_size) == 2:
                if isinstance(bkg_mesh_size[0], (int, float)) and isinstance(
                    bkg_mesh_size[1], (int, float)
                ):
                    self.bkg_mesh_size = (
                        bkg_mesh_size[0] * u.arcsec,
                        bkg_mesh_size[1] * u.arcsec,
                    )
                elif (
                    isinstance(bkg_mesh_size[0], u.Quantity)
                    and bkg_mesh_size[0].unit.is_equivalent(u.arcsec)
                    and isinstance(bkg_mesh_size[1], u.Quantity)
                    and bkg_mesh_size[1].unit.is_equivalent(u.arcsec)
                ):
                    self.bkg_mesh_size = bkg_mesh_size
                else:
                    raise ValueError(
                        "bkg_mesh_size is inhomogeneous. it should be either plain numbers or astropy.units.Quantity."
                    )
            else:
                raise ValueError(
                    "bkg_mesh_size must be a single quantity or a tuple of two quantities."
                )
        else:
            self.bkg_mesh_size = None
        if filters is None:
            if self.instrument is not None:
                self.filters = self.instrument.filters
            else:
                raise ValueError(
                    "Either filters or instrument must be specified to determine the filters."
                )
        else:
            if self.instrument is not None:
                if set(filters).issubset(self.instrument.filters):
                    self.filters = filters
                else:
                    raise ValueError(
                        "The filters provided are not available for the specified instrument."
                    )
            else:
                self.filters = filters
        self._swarp_extra_args = self._parse_swarp_args(kwargs)

    def combine(self):
        for filter in np.atleast_1d(self.filters):
            self.combine_per_filter(filter)

    @abstractmethod
    def combine_per_filter(self, filter):
        pass

    def _parse_swarp_args(self, kwargs):
        kwargs = {k.upper(): v for k, v in kwargs.items()}
        swarp_args = []
        for key, value in kwargs.items():
            if key in ["CENTER", "IMAGE_SIZE"]:
                raise ValueError(
                    f"{key} should be supplied as input arguments to {self.__class__.__name__}"
                )
            swarp_args.extend([f"-{key}", str(value)])
        if self.cutout_size is not None:
            if "PIXEL_SCALE" in kwargs:
                pixel_scale = kwargs["PIXEL_SCALE"]
            else:
                pixel_scale = self.instrument.pix_scale
            cutout_size_pix = [
                round(cutsize.to(u.arcsec).value / pixel_scale)
                for cutsize in self.cutout_size
            ]
            swarp_args.extend(
                ["-IMAGE_SIZE", f"{cutout_size_pix[0]},{cutout_size_pix[1]}"]
            )
        if self.cutout_cen is not None:
            swarp_args.extend(
                [
                    "-CENTER",
                    f"{self.cutout_cen.to_string('hmsdms', sep=':', precision=2)}",
                    "-CENTER_TYPE",
                    "MANUAL",
                ]
            )
        return swarp_args

    def _get_obs_ids(self):
        da = DataAccess(release_name=self.release_name)
        radius = 0.5 * max(*self.cutout_size)
        obs_ids = da.find_observations_for_target(
            self.cutout_cen.ra, self.cutout_cen.dec, radius, fully_contained=False
        )
        print(f"Determined required observation ids: {obs_ids}")
        return obs_ids

    @abstractmethod
    def _find_images(self):
        pass

    @abstractmethod
    def _prepare_input_for_swarp(self):
        pass

    @abstractmethod
    def _run_swarp(self):
        pass

    @abstractmethod
    def _post_process(self):
        pass

In [ ]:
# | export
# subclass for combing NISP images


class NISPCombiner(Combiner):
    """Combine NISP images using SWarp"""

    def __init__(self, **kwargs):
        super().__init__(instrument=NISP, swarp_config=SWARP_CONFIG_NISP, **kwargs)

    def combine(self):
        return super().combine()

    def combine_per_filter(self, filter):
        images = self._find_images(filter)
        if len(images) == 0:
            return
        out_fn = images[0].name.split("-")[0] + "-STK-IMAGE_" + filter
        if len(self.name) > 0:
            out_fn += f"-{self.name}"
        out_fn += ".fits"
        with tempfile.TemporaryDirectory(delete=(not self.debug)) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            self._prepare_input_for_swarp(images, tmpdir)
            self._run_swarp(tmpdir)
            self._post_process(tmpdir, out_fn)

    def _find_images(self, filter):
        """return a list of paths to the NISP images"""
        images = []
        for obs_id in np.atleast_1d(self.obs_ids):
            images.extend(
                self.in_path.glob(f"**/EUC_NIR*CAL-IMAGE_{filter}-{obs_id}-*Z.fits")
            )
        n_dithers = len(images)
        n_dither_per_obsid = 4  # four images per obsid for NISP
        n_obs = len(self.obs_ids)
        if n_dithers == 0:
            print("Found no files.")
        elif n_dithers < n_dither_per_obsid * n_obs:
            print("Missing some files.")
        elif n_dithers > n_dither_per_obsid * n_obs:
            print("Found too many files.")
        return images

    def _prepare_input_for_swarp(self, images, tmpdir):
        print("Preparing science and weight images for swarp...")
        sci_fns = []
        for i, dither in enumerate(images):
            with fits.open(dither) as hdul:
                for ext in self.instrument.extnames:
                    sci_data = hdul[f"{ext}.SCI"].data
                    rms_data = hdul[f"{ext}.RMS"].data
                    dq_data = hdul[f"{ext}.DQ"].data
                    primary_hdr = hdul[0].header
                    sci_ext_hdr = hdul[f"{ext}.SCI"].header
                    rms_ext_hdr = hdul[f"{ext}.RMS"].header
                    bad_pix_mask = np.any(
                        [
                            (dq_data & 2**bit > 0)
                            for bit in self.instrument.bad_pix_bits
                        ],
                        axis=0,
                    )
                    # subtract background if requested
                    if self.bkg_sub:
                        obj_mask, _ = fast_mask(sci_data, estimate_background=True)
                        # combine the bad pixel mask and the object mask to form a final mask
                        mask = bad_pix_mask | obj_mask
                        # 10 x 10 mesh by default if not specified
                        if self.bkg_mesh_size is None:
                            bkg_mesh_size_pix = (
                                sci_data.shape[0] // 10,
                                sci_data.shape[1] // 10,
                            )
                        else:
                            bkg_mesh_size_pix = (
                                round_up_box_size(
                                    sci_data.shape[0],
                                    self.bkg_mesh_size[0].to(u.arcsec).value
                                    / self.instrument.pix_scale,
                                ),
                                round_up_box_size(
                                    sci_data.shape[1],
                                    self.bkg_mesh_size[0].to(u.arcsec).value
                                    / self.instrument.pix_scale,
                                ),
                            )
                        bg = Background2D(
                            sci_data,
                            bkg_mesh_size_pix,
                            mask=mask,
                            filter_size=(self.filter_size, self.filter_size),
                            sigma_clip=SigmaClip(sigma=3),
                            exclude_percentile=90.0,
                        )
                        # subtract the background
                        sci_data -= bg.background
                        # compute FLXSCALE for swarp
                        exptime = primary_hdr["EXPTIME"]
                        photfnu = primary_hdr["PHOTFNU"]
                        phrelex = primary_hdr["PHRELEX"]
                        phreldt = sci_ext_hdr["PHRELDT"]
                        phoscale = (1.0 / exptime) * photfnu * phrelex * phreldt
                        sci_ext_hdr.append(
                            (
                                "PHOSCALE",
                                phoscale,
                                "Combined photometric scaling factors",
                            )
                        )
                    # compute weight map
                    with np.errstate(divide="ignore"):
                        np.divide(1.0, np.square(rms_data), out=rms_data)
                    # set the weight to 0 for bad pixels
                    rms_data[bad_pix_mask] = 0
                    sci_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(sci_data, sci_ext_hdr),
                        ]
                    )
                    weight_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(rms_data, rms_ext_hdr),
                        ]
                    )
                    sci_fn = f"sci_{i}_{ext}.fits"
                    wt_fn = f"sci_{i}_{ext}.weight.fits"
                    sci_fns.append(sci_fn)
                    sci_hdul.writeto(tmpdir / sci_fn)
                    weight_hdul.writeto(tmpdir / wt_fn)
        # prepare image list for swarp
        with open(tmpdir / "images.list", "w") as f:
            for fn in sci_fns:
                f.write(f"{fn}[1]\n")

    def _run_swarp(self, tmpdir):
        print("Running SWarp...")
        with open(tmpdir / "config.swarp", "w") as f:
            f.write(self.swarp_config)
        swarp_cmd = ["swarp", "@images.list", "-c", "config.swarp"]
        swarp_cmd.extend(self._swarp_extra_args)
        try:
            run_swarp = subprocess.run(
                swarp_cmd,
                cwd=tmpdir,
                text=True,
                capture_output=True,
                check=True,
            )
            if self.debug:
                print(run_swarp.stderr)
            print("SWarp finished successfully.")
        except subprocess.CalledProcessError as e:
            print(
                f"Command '{' '.join(e.cmd)}' returned non-zero exit status, please check its stderr below."
            )
            print(e.stderr)

    def _post_process(self, tmpdir, out_fn):
        """clean up FITS headers and copy the output to the desired directory"""
        with fits.open(tmpdir / "coadd.fits", memmap=True) as hdul_sci, fits.open(
            tmpdir / "coadd.weight.fits", memmap=True
        ) as hdul_weights:
            # this is necessary because they are both PrimaryHDU objects
            hdu_sci = fits.ImageHDU(hdul_sci[0].data, hdul_sci[0].header)
            hdu_rms = fits.ImageHDU(hdul_weights[0].data, hdul_weights[0].header)
            # set science image to NaN where the weight is zero
            hdu_sci.data[hdu_rms.data == 0] = np.nan
            # convert the weight to RMS
            with np.errstate(divide="ignore"):
                np.divide(1.0, hdu_rms.data, out=hdu_rms.data)
            np.sqrt(hdu_rms.data, out=hdu_rms.data)
            # clean up the headers
            hdu_sci.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_sci.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_sci.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_sci.header.set("EXTNAME", "SCI", "Extension name", after="GCOUNT")
            hdu_sci.header.set("ZPAB", 23.9)
            hdu_sci.header.set("ZPABE", 0.0)
            hdu_sci.header.set("ZPVEGA", 1.0)
            hdu_sci.header.set("ZPVEGAE", 0.0)
            hdu_sci.header.remove("SIMPLE", ignore_missing=True)
            hdu_rms.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_rms.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_rms.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_rms.header.set("EXTNAME", "RMS", "Extension name", after="GCOUNT")
            for key in ("SIMPLE", "ZPAB", "ZPAB", "ZPVEGA", "ZPVEGAE"):
                hdu_rms.header.remove(key, ignore_missing=True)
            hdul = fits.HDUList([fits.PrimaryHDU(), hdu_sci, hdu_rms])
            self.out_dir.mkdir(parents=True, exist_ok=True)
            hdul.writeto(self.out_dir / out_fn, overwrite=self.overwrite)

In [ ]:
# | export


class VISCombiner(Combiner):
    """Combine VIS images using SWarp"""

    def __init__(self, **kwargs):
        super().__init__(instrument=VIS, swarp_config=SWARP_CONFIG_VIS, **kwargs)

    def combine(self):
        return super().combine()

    def combine_per_filter(self, filter):
        images = self._find_images()
        if len(images) == 0:
            return
        out_fn = images[0].name.split("-")[0] + "-STK_" + filter
        if len(self.name) > 0:
            out_fn += f"-{self.name}"
        out_fn += ".fits"
        with tempfile.TemporaryDirectory(delete=(not self.debug)) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            self._prepare_input_for_swarp(images, tmpdir)
            self._run_swarp(tmpdir)
            self._post_process(tmpdir, out_fn)

    def _find_images(self):
        """return a list of paths to the VIS images"""
        images = []
        for obs_id in np.atleast_1d(self.obs_ids):
            images.extend(self.in_path.glob(f"**/EUC_VIS_SWL-DET-*{obs_id}-*Z.fits"))
        n_dithers = len(images)
        n_dither_per_obsid = 6  # number of dithers per obsid for NISP
        n_obs = len(self.obs_ids)
        if n_dithers == 0:
            print("Found no files.")
        elif n_dithers < n_dither_per_obsid * n_obs:
            print("Missing some files.")
        elif n_dithers > n_dither_per_obsid * n_obs:
            print("Found too many files.")
        return images

    def _prepare_input_for_swarp(self, images, tmpdir):
        print("Preparing science and weight images for swarp...")
        sci_fns = []
        for i, dither in enumerate(images):
            with fits.open(dither) as hdul:
                for ext in self.instrument.extnames:
                    sci_data = hdul[f"{ext}.SCI"].data
                    rms_data = hdul[f"{ext}.RMS"].data
                    dq_data = hdul[f"{ext}.FLG"].data
                    primary_hdr = hdul[0].header
                    sci_ext_hdr = hdul[f"{ext}.SCI"].header
                    rms_ext_hdr = hdul[f"{ext}.RMS"].header
                    bad_pix_mask = np.any(
                        [
                            (dq_data & 2**bit > 0)
                            for bit in self.instrument.bad_pix_bits
                        ],
                        axis=0,
                    )
                    # subtract background if requested
                    if self.bkg_sub:
                        obj_mask, _ = fast_mask(sci_data, estimate_background=True)
                        # combine the bad pixel mask and the object mask to form a final mask
                        mask = bad_pix_mask | obj_mask
                        # 3 x 3 mesh by default if not specified
                        if self.bkg_mesh_size is None:
                            bkg_mesh_size_pix = (
                                sci_data.shape[0] // 3,
                                sci_data.shape[1] // 3,
                            )
                        else:
                            bkg_mesh_size_pix = (
                                round_up_box_size(
                                    sci_data.shape[0],
                                    self.bkg_mesh_size[0].to(u.arcsec).value
                                    / self.instrument.pix_scale,
                                ),
                                round_up_box_size(
                                    sci_data.shape[1],
                                    self.bkg_mesh_size[0].to(u.arcsec).value
                                    / self.instrument.pix_scale,
                                ),
                            )
                        bg = Background2D(
                            sci_data,
                            bkg_mesh_size_pix,
                            mask=mask,
                            filter_size=(self.filter_size, self.filter_size),
                            sigma_clip=SigmaClip(sigma=3),
                            exclude_percentile=90.0,
                        )
                        # subtract the background
                        sci_data -= bg.background
                    # compute weight map
                    with np.errstate(divide="ignore"):
                        np.divide(1.0, np.square(rms_data), out=rms_data)
                    # set the weight to 0 for bad pixels
                    rms_data[bad_pix_mask] = 0
                    sci_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(sci_data, sci_ext_hdr),
                        ]
                    )
                    weight_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(rms_data, rms_ext_hdr),
                        ]
                    )
                    sci_fn = f"sci_{i}_{ext}.fits"
                    wt_fn = f"sci_{i}_{ext}.weight.fits"
                    sci_fns.append(sci_fn)
                    sci_hdul.writeto(tmpdir / sci_fn)
                    weight_hdul.writeto(tmpdir / wt_fn)
        # prepare image list for swarp
        with open(tmpdir / "images.list", "w") as f:
            for fn in sci_fns:
                f.write(f"{fn}[1]\n")

    def _run_swarp(self, tmpdir):
        print("Running SWarp...")
        with open(tmpdir / "config.swarp", "w") as f:
            f.write(self.swarp_config)
        swarp_cmd = ["swarp", "@images.list", "-c", "config.swarp"]
        swarp_cmd.extend(self._swarp_extra_args)
        try:
            run_swarp = subprocess.run(
                swarp_cmd,
                cwd=tmpdir,
                text=True,
                capture_output=True,
                check=True,
            )
            if self.debug:
                print(run_swarp.stderr)
            print("SWarp finished successfully.")
        except subprocess.CalledProcessError as e:
            print(
                f"Command '{' '.join(e.cmd)}' returned non-zero exit status, please check its stderr below."
            )
            print(e.stderr)

    def _post_process(self, tmpdir, out_fn):
        """clean up FITS headers and copy the output to the desired directory"""
        with fits.open(tmpdir / "coadd.fits", memmap=True) as hdul_sci, fits.open(
            tmpdir / "coadd.weight.fits", memmap=True
        ) as hdul_weights:
            # this is necessary because they are both PrimaryHDU objects
            hdu_sci = fits.ImageHDU(hdul_sci[0].data, hdul_sci[0].header)
            hdu_rms = fits.ImageHDU(hdul_weights[0].data, hdul_weights[0].header)
            # set science image to NaN where the weight is zero
            hdu_sci.data[hdu_rms.data == 0] = np.nan
            # convert the weight to RMS
            with np.errstate(divide="ignore"):
                np.divide(1.0, hdu_rms.data, out=hdu_rms.data)
            np.sqrt(hdu_rms.data, out=hdu_rms.data)
            # clean up the headers
            hdu_sci.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_sci.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_sci.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_sci.header.set("EXTNAME", "SCI", "Extension name", after="GCOUNT")
            hdu_sci.header.set("ZPAB", 23.9)
            hdu_sci.header.set("ZPABE", 0.0)
            hdu_sci.header.set("ZPVEGA", 1.0)
            hdu_sci.header.set("ZPVEGAE", 0.0)
            hdu_sci.header.remove("SIMPLE", ignore_missing=True)
            hdu_rms.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_rms.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_rms.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_rms.header.set("EXTNAME", "RMS", "Extension name", after="GCOUNT")
            for key in ("SIMPLE", "ZPAB", "ZPAB", "ZPVEGA", "ZPVEGAE"):
                hdu_rms.header.remove(key, ignore_missing=True)
            hdul = fits.HDUList([fits.PrimaryHDU(), hdu_sci, hdu_rms])
            self.out_dir.mkdir(parents=True, exist_ok=True)
            hdul.writeto(self.out_dir / out_fn, overwrite=self.overwrite)

## Examples

Produce a H-band stack for single Q1 observation.

In [ ]:
in_dir_NISP = default_data_path("Q1_R1")
obs_id = 2682
out_dir = default_data_path("Q1_R1_processed_test", f"stacked/{obs_id}")

In [ ]:
%%time
combiner = Combiner(in_path=in_path, out_path=out_path, obs_ids=obs_id, overwrite=True)
combiner.combine(filters="H")

Produce a H-band stack image of a galaxy cluster from Euclid Q1 data.

First, with an angular size that is contained within a single observation.

In [ ]:
cluster_id = "MCXCJ1754.6+6803"
z = 0.07
ra = 268.662 * u.deg
dec = 68.058 * u.deg
cutout_radius_physical = 500 * u.kpc
cluster_path = default_data_path("Q1_R1_clusters_test", cluster_id)
cutout_radius_angular = physical_to_angular(cutout_radius_physical, z)
cutout_size = 2 * cutout_radius_angular
bkg_mesh_size = 0.5 * cutout_radius_angular

In [ ]:
# here we use original data from the archive an save to a test location
in_dir = default_data_path("Q1_R1")
out_dir = default_data_path("Q1_R1_clusters_test", cluster_id)

# to use processed data of a particular version and processing stage
# in_path = default_data_path("Q1_R1_processed_v0.2", "persistence")
# out_path = default_data_path("Q1_R1_clusters_v0.2", cluster_id)

In [ ]:
combiner = Combiner(
    in_path=in_dir,
    out_path=out_dir,
    bkg_mesh_size=bkg_mesh_size,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=cluster_id,
    overwrite=True,
)
combiner.combine(filters="H")

Second, with an angular size that requires multiple observations (2682, 2683, 2685) to be combined.

In [ ]:
cutout_size = 6 * cutout_radius_angular
bkg_mesh_size = 0.5 * cutout_radius_angular
name = f"{cluster_id}-big"

In [ ]:
combiner = Combiner(
    in_path=in_dir,
    out_path=out_dir,
    bkg_mesh_size=bkg_mesh_size,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=name,
    overwrite=True,
)
combiner.combine(filters="H")

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()